# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup and the label

*Clones my repo on Colab, then defines the two time windows and the outcome I'm trying to predict.*

In [1]:
import os, sys, subprocess, json, warnings
import numpy as np
import pandas as pd
import sklearn
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ARS-0/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
os.makedirs("work/outputs", exist_ok=True)

SEED = 42
np.random.seed(SEED)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape, "| scikit-learn", sklearn.__version__, "| seed", SEED)

Rows, columns: (30000, 44) | scikit-learn 1.6.1 | seed 42


In [2]:
# ---- Two windows inside the 90-day snapshot --------------------------------
# OUTCOME window  = the most recent 30 days (*_last_30d)
# FEATURE window  = everything older: days 31-90 back = (90d total) - (last 30d)
# The 90-day columns (ctr, avg_position, impressions_90d ...) CONTAIN the outcome window,
# so I only use them after subtracting it out.
df["pre_impr"] = df["impressions_90d"] - df["impressions_last_30d"]
df["pre_clk"]  = df["clicks_90d"] - df["clicks_last_30d"]
assert (df["pre_impr"] >= 0).all() and (df["pre_clk"] >= 0).all()

# Eligible pages: enough volume in BOTH windows, and a real position (0 = no data)
MIN_IMPR = 500
e = df[(df["pre_impr"] >= MIN_IMPR) & (df["impressions_last_30d"] >= MIN_IMPR) & (df["avg_position"] > 0)].reset_index(drop=True)
print(f"Eligible pages: {len(e):,} of {len(df):,}  |  clients: {e['client_id'].nunique()}")

# ---- Label: an OBSERVED future outcome --------------------------------------
# y = 1 if, in the last 30 days, the page got at most HALF the clicks its tier peers got
# for the same impressions. Tier benchmark = pooled last-30d CTR of the tier.
bench = e.groupby("position_tier").apply(lambda x: x["clicks_last_30d"].sum() / x["impressions_last_30d"].sum())
e["expected_last30"] = e["position_tier"].map(bench) * e["impressions_last_30d"]
e["y"] = (e["clicks_last_30d"] <= 0.5 * e["expected_last30"]).astype(int)
print("\nOutcome benchmark (pooled CTR %, last 30 days):")
print((bench * 100).round(3).to_string())
print(f"\nBase rate (share of eligible pages with y = 1): {e['y'].mean():.3f}")
print("Median clicks a page was expected to get in the outcome window:", round(e["expected_last30"].median(), 1))
print("\nPages per client (top 5):"); print(e["client_id"].value_counts().head(5).to_string())

Eligible pages: 9,102 of 30,000  |  clients: 28

Outcome benchmark (pooled CTR %, last 30 days):
position_tier
deep        0.024
page_1      0.361
page_3_5    0.220
striking    0.429
top_3       0.570

Base rate (share of eligible pages with y = 1): 0.361
Median clicks a page was expected to get in the outcome window: 5.9

Pages per client (top 5):
client_id
client_19581e27de    3162
client_6208ef0f77    2119
client_4e07408562    1266
client_3fdba35f04     549
client_7f2253d7e2     438


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Question and label

My lane is CTR / Engagement Opportunity Scoring. The question: **using what a page did in days 31-90 back, can I predict which pages will under-capture clicks in the last 30 days?**

The snapshot has no separate future file, so I split its 90 days in two. The **last 30 days** are the outcome. Days 31-90 are the features, found by subtracting the last-30-day numbers out of the 90-day totals. The label `y = 1` means: in the outcome window the page got at most **half** the clicks its position-tier peers got for the same impressions. It is an observed outcome, not a current bucket and not a product flag. I kept pages with at least 500 impressions in *both* windows, so a handful of clicks can't flip the label. That leaves 9,102 pages across 28 clients, with a base rate of 36%.

### Method choice

- **Baseline:** my Week-4 rule (missed clicks vs tier peers), re-run on the feature window only. The W04 version used all 90 days, which overlaps the outcome, so it is fine for a snapshot queue but not for scoring. Same rule, new window.
- **Logistic Regression first:** the question is "which pages first," with an observed yes/no label, so a readable model comes first. I can read its coefficients.
- **Random Forest second:** to see whether a stronger model earns anything over Logistic Regression.
- **Decision tree, depth 3:** a model I can print and read in full.
- **Metric:** precision@50, @100 and @200, because the output is a ranked review queue. I also print the base rate and AUC.

I skipped gradient boosting. With 9k rows and one dominant signal, I didn't expect it to earn its complexity.

### Features and what I kept out

Features come from the feature window (pre-window impressions, clicks and CTR, plus the 30 days before the outcome) and static page facts (content type, intent, search volume, competition, CPC, word count, age). Missing values get a flag (`has_word_count`, `has_volume`) rather than a silent zero.

Kept out: `trend_direction`, `trend_pct`, `is_declining_label`, every `_last_30d` column, the 90-day totals and `ctr`, because they contain the outcome. `days_since_last_update` is also out, since a rewrite during the outcome window could change the result. **One honest limit:** `avg_position` and `position_tier` cover all 90 days, so they overlap the outcome window. I kept them because a position-adjusted CTR needs them, and section 4 tests what happens without them.

In [3]:
# ---- My Week-4 baseline, re-run on the FEATURE window only -------------------
# Same rule as W04: score = missed clicks vs tier peers. The only change is the window:
# W04 used the full 90 days (fine for a snapshot queue, but it overlaps the outcome).
pre_rate = e.groupby("position_tier").apply(lambda x: x["pre_clk"].sum() / x["pre_impr"].sum())
e["baseline_score"] = (e["position_tier"].map(pre_rate) * e["pre_impr"] - e["pre_clk"]).clip(lower=0)

# ---- Features: all from the feature window or static page facts ---------------
e["log_pre_impr"]    = np.log1p(e["pre_impr"])
e["log_pre_clk"]     = np.log1p(e["pre_clk"])
e["pre_ctr_pct"]     = e["pre_clk"] / e["pre_impr"] * 100
e["log_prev30_impr"] = np.log1p(e["impressions_prev_30d"])
e["prev30_ctr_pct"]  = np.where(e["impressions_prev_30d"] > 0, e["clicks_prev_30d"] / e["impressions_prev_30d"].clip(lower=1) * 100, 0)
e["has_word_count"]  = e["word_count"].notna().astype(int)           # missingness follows content_type -> flag it
e["word_count_f"]    = e["word_count"].fillna(e["word_count"].median())
e["has_volume"]      = e["search_volume"].notna().astype(int)
e["log_volume"]      = np.log1p(e["search_volume"].fillna(0))
e["competition_f"]   = e["competition"].fillna(0)
e["cpc_f"]           = e["cpc"].fillna(0)
e["log_age"]         = np.log1p(e["content_age_days"])

NUM = ["log_pre_impr", "log_pre_clk", "pre_ctr_pct", "log_prev30_impr", "prev30_ctr_pct",
       "avg_position", "log_volume", "competition_f", "cpc_f", "word_count_f",
       "has_word_count", "has_volume", "log_age"]
CAT = ["content_type", "main_intent", "position_tier"]
X = pd.concat([e[NUM], pd.get_dummies(e[CAT], dtype=int)], axis=1)
y = e["y"].values
groups = e["client_id"].values

FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label", "health_score", "priority_score", "action_type",
             "ctr", "impressions_90d", "clicks_90d", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]
leaks = [c for c in X.columns if c in FORBIDDEN or "last_30d" in c]
print("Feature count:", X.shape[1])
print("Forbidden / outcome-window columns found among features:", leaks)
assert not leaks
print("\nCaveat on avg_position / position_tier: they are measured over all 90 days, so they include the outcome window.")
print("I keep them (the lane needs position-adjusted CTR) and test what happens without them in section 4.")

Feature count: 25
Forbidden / outcome-window columns found among features: []

Caveat on avg_position / position_tier: they are measured over all 90 days, so they include the outcome window.
I keep them (the lane needs position-adjusted CTR) and test what happens without them in section 4.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'm splitting **by client** (5-fold GroupKFold). A page's neighbours at the same client share a site, an audience and a CTR habit, so a random split would let the model see the client in training and look better than it is. Grouping by client asks the real question: *would this work on a client it has never seen?* The code asserts that no client appears on both sides of any fold.

Two things to know. The split is not time-aware, because the data is one snapshot and the outcome window is already the later one. And the folds are lopsided: **client 19581e27de alone has 35% of the pages and ends up as fold 0 by itself, and the second-biggest client is fold 1 by itself.** Those folds are a test on one client each. That's why I report every fold and not just the average. The baseline is scored on the exact same test pages as the models, and all random seeds are fixed at 42.

In [4]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(X, y, groups))
rows = []
for i, (tr, te) in enumerate(folds):
    overlap = set(groups[tr]) & set(groups[te])
    assert not overlap                       # no client appears on both sides
    rows.append({"fold": i, "train_pages": len(tr), "test_pages": len(te),
                 "test_clients": len(set(groups[te])), "test_base_rate": round(y[te].mean(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nNo client is in both train and test in any fold.")

 fold  train_pages  test_pages  test_clients  test_base_rate
    0         5940        3162             1           0.464
    1         6983        2119             1           0.257
    2         7828        1274             4           0.354
    3         7828        1274            11           0.382
    4         7829        1273            11           0.261

No client is in both train and test in any fold.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

KS = [50, 100, 200]
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return np.asarray(labels)[order].mean()

def make_models():
    return {
        "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED),
        "Logistic Regression":     make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5)),
        "Random Forest":           RandomForestClassifier(n_estimators=300, min_samples_leaf=20, max_features="sqrt",
                                                          random_state=SEED, n_jobs=-1),
    }

def run_cv(Xmat, label="full"):
    """Return per-fold metric rows and out-of-fold predictions for every method."""
    per_fold, oof = [], {}
    for i, (tr, te) in enumerate(folds):
        yt = y[te]
        scores = {"Baseline rule (W04 logic)": e["baseline_score"].values[te]}
        for name, m in make_models().items():
            m.fit(Xmat.iloc[tr], y[tr])
            scores[name] = m.predict_proba(Xmat.iloc[te])[:, 1]
        for name, s in scores.items():
            r = {"fold": i, "method": name, "auc": roc_auc_score(yt, s)}
            for k in KS: r[f"p@{k}"] = precision_at_k(s, yt, k)
            per_fold.append(r)
            oof.setdefault(name, np.zeros(len(y)))[te] = s
    return pd.DataFrame(per_fold), oof

per_fold, oof = run_cv(X)
base_rates = np.array([y[te].mean() for _, te in folds])

summary = per_fold.groupby("method").agg(**{f"precision@{k}": (f"p@{k}", "mean") for k in KS}, AUC=("auc", "mean"))
summary = summary.loc[["Baseline rule (W04 logic)", "Decision tree (depth 3)", "Logistic Regression", "Random Forest"]]
summary["lift_vs_base@50"] = summary["precision@50"] / base_rates.mean()
print(f"MODEL vs BASELINE  (5-fold, split by client; mean over folds; base rate = {base_rates.mean():.3f})")
print(summary.round(3).to_string())

print("\nPrecision@50 by fold (min / max shows how much it moves):")
pf = per_fold.pivot(index="method", columns="fold", values="p@50").loc[summary.index]
pf["min"], pf["max"] = pf.min(axis=1), pf.max(axis=1)
print(pf.round(2).to_string())
print("\nFolds where each model beats the baseline at precision@50:")
for m in summary.index[1:]:
    print(f"  {m}: {(pf.loc[m, [0,1,2,3,4]].values > pf.loc['Baseline rule (W04 logic)', [0,1,2,3,4]].values).sum()} of 5")

summary.round(4).reset_index().to_csv("work/outputs/w05_model_vs_baseline.csv", index=False)

MODEL vs BASELINE  (5-fold, split by client; mean over folds; base rate = 0.344)
                           precision@50  precision@100  precision@200    AUC  lift_vs_base@50
method                                                                                       
Baseline rule (W04 logic)         0.704          0.636          0.588  0.738            2.049
Decision tree (depth 3)           0.644          0.680          0.666  0.789            1.874
Logistic Regression               0.800          0.764          0.742  0.823            2.328
Random Forest                     0.816          0.818          0.752  0.834            2.375

Precision@50 by fold (min / max shows how much it moves):
fold                          0     1     2     3     4   min   max
method                                                             
Baseline rule (W04 logic)  0.84  0.72  0.64  0.76  0.56  0.56  0.84
Decision tree (depth 3)    0.84  0.48  0.66  0.66  0.58  0.48  0.84
Logistic Regression     

### Reading the table

| | precision@50 | @100 | @200 | AUC |
|---|---|---|---|---|
| Baseline rule | 0.70 | 0.64 | 0.59 | 0.74 |
| Decision tree (depth 3) | 0.64 | 0.68 | 0.67 | 0.79 |
| Logistic Regression | 0.80 | 0.76 | 0.74 | 0.82 |
| Random Forest | 0.82 | 0.82 | 0.75 | 0.83 |

Base rate is 0.34 on average across folds, so random picking would get about 1 in 3 right.

- **Both Logistic Regression and Random Forest beat my baseline** at every K, by roughly 10 points at precision@50. Each beats it in 4 of 5 folds.
- **Random Forest is only a little better than Logistic Regression** (0.82 vs 0.80 at @50). That gap is smaller than the fold-to-fold swing (for example 0.66 to 0.92 for Logistic Regression). So the simple, readable model gets nearly all of the gain, and I would hand over Logistic Regression.
- **The depth-3 tree lost to the baseline at @50** and wins only 2 of 5 folds. It beats the baseline at @100 and @200, so the picture is mixed and I'm reporting it that way.
- With only 50 picks per fold, a 0.05 difference is noise. I treat the ranking among the models as directional, and only the gap between the baseline and the two stronger models as a pattern.
- The baseline was never built to predict this label. It ranks by missed clicks, which favours big pages, while the models rank by chance of under-capturing. The comparison is fair (same pages, same label, same metric), but it's also why the baseline's AUC is lower.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# ---- (a) Sensitivity: take away the two columns that overlap the outcome window ----
drop_cols = [c for c in X.columns if c == "avg_position" or c.startswith("position_tier_")]
X_nopos = X.drop(columns=drop_cols)
pf2, _ = run_cv(X_nopos, "no position")
s2 = pf2[pf2["method"] != "Baseline rule (W04 logic)"].groupby("method").agg(**{f"precision@{k}": (f"p@{k}", "mean") for k in KS}, AUC=("auc", "mean"))
print("Models WITHOUT avg_position / position_tier (rule baseline needs the tier, so it is not repeated):")
print(s2.round(3).to_string())

# ---- (b) What does the model lean on? Permutation importance on held-out clients ----
from sklearn.inspection import permutation_importance
imp = []
for tr, te in folds:
    rf = make_models()["Random Forest"].fit(X.iloc[tr], y[tr])
    r = permutation_importance(rf, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
    imp.append(r.importances_mean)
imp = pd.Series(np.mean(imp, axis=0), index=X.columns).sort_values(ascending=False)
print("\nTop 8 features by permutation importance (drop in AUC when shuffled, held-out folds):")
print(imp.head(8).round(4).to_string())

# LogReg direction of the top features (standardised coefficients, fit on everything, for reading only)
lr = make_models()["Logistic Regression"].fit(X, y)
coef = pd.Series(lr[-1].coef_[0], index=X.columns).sort_values()
print("\nLogistic Regression: strongest pushes toward y = 1 (under-capturing clicks) and away from it")
print(pd.concat([coef.tail(4), coef.head(4)]).round(3).to_string())

# ---- the readable tree ----
from sklearn.tree import export_text
tree = make_models()["Decision tree (depth 3)"].fit(X, y)
print("\nDepth-3 decision tree (fit on all pages, for reading only):")
print(export_text(tree, feature_names=list(X.columns)))

Models WITHOUT avg_position / position_tier (rule baseline needs the tier, so it is not repeated):
                         precision@50  precision@100  precision@200    AUC
method                                                                    
Decision tree (depth 3)         0.644          0.680          0.666  0.789
Logistic Regression             0.784          0.754          0.720  0.813
Random Forest                   0.812          0.772          0.732  0.824

Top 8 features by permutation importance (drop in AUC when shuffled, held-out folds):
pre_ctr_pct               0.0699
prev30_ctr_pct            0.0564
log_pre_clk               0.0171
avg_position              0.0049
log_prev30_impr           0.0040
position_tier_page_3_5    0.0022
log_age                   0.0016
log_pre_impr              0.0010

Logistic Regression: strongest pushes toward y = 1 (under-capturing clicks) and away from it
position_tier_striking    0.220
position_tier_page_1      0.283
avg_position     

In [8]:
# ---- (c) Where is the best model wrong? (out-of-fold predictions, so every page was scored by a model that never saw its client)
best = "Random Forest"
e["p_best"] = oof[best]
e["p_base"] = oof["Baseline rule (W04 logic)"]
top = e.sort_values("p_best", ascending=False).head(200)
print(f"Top-200 by {best}: precision {top['y'].mean():.3f}  (base rate {y.mean():.3f})")

print("\nPrecision of the model's top 200 by position tier, and by content type (n printed):")
print(top.groupby("position_tier")["y"].agg(n="size", precision="mean").round(3).to_string())
print(top.groupby("content_type")["y"].agg(n="size", precision="mean").round(3).to_string())

print("\nAll eligible pages, by content type: how often is y = 1, and how many are there?")
print(e.groupby("content_type")["y"].agg(n="size", share_y1="mean").round(3).to_string())

# Is the model's top-200 just the biggest client?
print("\nClients in the model's top 200:", top["client_id"].nunique(), "| biggest client's share:",
      round(top["client_id"].value_counts(normalize=True).iloc[0], 2))

# Three concrete wrong cases: highest-scored pages that did NOT under-capture
wrong = top[top["y"] == 0].head(3)
cols = ["content_id", "content_type", "position_tier", "avg_position", "pre_impr", "pre_clk",
        "impressions_last_30d", "clicks_last_30d", "expected_last30", "p_best"]
print("\nThree confident misses (model said under-capturing, outcome window says no):")
print(wrong[cols].round(2).to_string(index=False))
print("\nTheir CTR before vs after (%):")
print(pd.DataFrame({"content_id": wrong["content_id"],
                    "pre_ctr_pct": (wrong["pre_clk"] / wrong["pre_impr"] * 100).round(3),
                    "outcome_ctr_pct": (wrong["clicks_last_30d"] / wrong["impressions_last_30d"] * 100).round(3),
                    "tier_benchmark_pct": (wrong["position_tier"].map(bench) * 100).round(3)}).to_string(index=False))

# ---- save receipts ----
metrics = {"eligible_pages": int(len(e)), "base_rate": round(float(y.mean()), 4), "cv": "GroupKFold(5) by client_id",
           "seed": SEED, "sklearn": sklearn.__version__,
           "model_vs_baseline": json.loads(summary.round(4).to_json(orient="index"))}
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nSaved work/outputs/w05_model_metrics.json and w05_model_vs_baseline.csv")

Top-200 by Random Forest: precision 0.875  (base rate 0.361)

Precision of the model's top 200 by position tier, and by content type (n printed):
                n  precision
position_tier               
deep           34      0.853
page_1         74      0.905
page_3_5       37      0.865
striking       55      0.855
                   n  precision
content_type                   
keyword article  200      0.875

All eligible pages, by content type: how often is y = 1, and how many are there?
                       n  share_y1
content_type                      
comparison article     8     0.625
feedly article        38     0.368
keyword article     9056     0.360

Clients in the model's top 200: 13 | biggest client's share: 0.44

Three confident misses (model said under-capturing, outcome window says no):
          content_id    content_type position_tier  avg_position  pre_impr  pre_clk  impressions_last_30d  clicks_last_30d  expected_last30  p_best
content_08eb507fdfc2 keyword artic

### What the model leans on

Permutation importance on held-out clients puts two features far ahead of everything else: the page's **own CTR in the feature window** (`pre_ctr_pct`, 0.070) and its CTR in the 30 days before the outcome (`prev30_ctr_pct`, 0.056). Next is clicks in the feature window (0.017). Position, volume, content type and word count barely register. The depth-3 tree says the same thing: it splits first on `pre_ctr_pct <= 0.21`, and the deeper splits never change the predicted class.

In plain words, **a page that under-captured clicks before keeps doing so.** That's believable, and it isn't leakage, because those columns come from before the outcome window. It also means the model is mostly learning persistence, not something new about titles or content. Logistic Regression agrees on direction: lower past CTR pushes toward `y = 1`. I read the coefficients as association only.

### Sensitivity check

Without `avg_position` and `position_tier` (the columns that overlap the outcome window), Random Forest goes from 0.816 to 0.812 at @50 and Logistic Regression from 0.800 to 0.784. The result barely moves, so the overlap isn't what's driving it.

### Where it is wrong

- **The model's top 200 are 87.5% correct** against a 36% base rate, and it holds up across tiers (85-90%). But 44% of those 200 pages belong to one client, and they come from 13 clients overall. So the queue is dominated by the biggest accounts.
- **Content type is untestable.** The eligible set has 9,056 keyword articles, 38 feedly articles and **8 comparison articles**. The model's top 200 are all keyword articles. I can't say anything about comparison articles here, so my Week-1 finding stays a finding from the larger, looser sample. That is a limit of this data slice, not a result.
- **Confident misses are near the line.** All three of my example misses are close calls. `content_08eb507fdfc2` is a deep-tier page that got 1 click against about 0.5 expected, so a single click made it "not under-capturing." `content_983adc986531` and `content_b1bc831deff1` had almost no clicks in the feature window, then recovered to roughly the tier benchmark (0.16% vs 0.22%, and 0.45% vs 0.43%). These are the hardest cases: tiny click counts, where one or two clicks flip the label.
- **Deep-tier labels are fragile.** The benchmark there is 0.02%, so "under half of expected" effectively means "zero clicks." I would trust those labels least.
- **Folds disagree.** Fold 0 (one client) is easy for every method, and fold 1 (another single client) is hardest. Results on a brand-new client can land anywhere in that range.

### Bottom line

On clients the model never saw, a readable Logistic Regression ranked next-month under-capturing pages about 10 points better than my hand rule at precision@50 (0.80 vs 0.70). Most of the gain comes from "low CTR before, low CTR after." That is a directional, decision-support result on one snapshot, not proof that rewriting a title will lift clicks. It does not show what causes low CTR.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.